# Lezione 07 — Analisi grammaticale ed entità

**Corso:** NLP Classico con Python · **Blocco 3:** Struttura e significato del testo
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/07_analisis_gramatical_entidades_IT.md`

## Cosa vedrai in questa lezione

- **POS** (categoria grammaticale) e **dipendenze sintattiche** con spaCy in spagnolo e in italiano: leggere una tabella di token e un albero.
- **Estrarre** «chi fa cosa» (soggetto, verbo, oggetto) percorrendo l'albero, e vedere dove sbaglia un modello piccolo.
- **NER**: entità nominate (PER, ORG, LOC, MISC) su un corpus di 67 frasi di notizie sintetiche annotate a mano per lingua.
- **Valutare** un NER: definire prima cos'è un successo (span ed etichetta esatti contro corrispondenza parziale), calcolare precisione, richiamo (*recall*) e F1 **a mano** e per entità.
- **Errori tipici** dei modelli piccoli e **regole** (`Matcher`, `EntityRuler`) per correggerli, misurando su frasi che non sono state usate per scriverle.

## Prerequisiti

- Tokenizzazione e lemmatizzazione (lezioni 2 e 3); precisione, richiamo e F1 come idee (la lezione 5 le riprende con più dettaglio).
- Python con `numpy`, `pandas`, `matplotlib` e `spacy` 3.8 con `es_core_news_sm` e `it_core_news_sm`. La cella seguente installa solo ciò che manca (anche in Google Colab).
- La cartella `datos/` del corso (`noticias_ner_es.csv` e `noticias_ner_it.csv`). In Colab, caricala dal pannello dei file.

> **Come usare questo notebook:** esegui le celle in ordine. Le previsioni di spaCy dipendono dalla versione del modello (qui, spaCy 3.8 con i modelli 3.8.0): le cifre sono un *esempio illustrativo* su un corpus piccolo e non si generalizzano. I calcoli che dipendono solo dal nostro codice si verificano con `assert`.

---
## Installazione

Installa solo ciò che manca (pacchetti e modelli). Se c'è già tutto, non fa nulla.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("spacy", "spacy")
print("Librerie pronte.")

---
## Cella 0 — Setup

Importiamo le librerie, fissiamo il **seme**, definiamo `cargar_datos` e `cargar_modelo` e carichiamo i due modelli piccoli. Annota le versioni che vengono stampate: le previsioni che vedrai dipendono da esse.

In [ ]:
import os
import re
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)                          # seme fisso
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

import spacy

MODELOS = {"spa": "es_core_news_sm", "ita": "it_core_news_sm"}

def cargar_modelo(nombre):
    """Carica un modello spaCy e lo scarica se manca (idempotente)."""
    try:
        return spacy.load(nombre)
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", "-q", nombre], check=True)
        return spacy.load(nombre)

MARCA = re.compile(r"\{([^{}|]+)\|(PER|ORG|LOC|MISC)\}")

def parsear(marcado):
    """Converte 'Ciao {Ana|PER}' in (testo, {(inizio, fine, etichetta)})."""
    texto, ents, pos = "", set(), 0
    for m in MARCA.finditer(marcado):
        texto += marcado[pos:m.start()]
        ents.add((len(texto), len(texto) + len(m.group(1)), m.group(2)))
        texto += m.group(1)
        pos = m.end()
    return texto + marcado[pos:], ents

NLP = {idioma: cargar_modelo(nombre) for idioma, nombre in MODELOS.items()}
ETIQUETAS = ["PER", "ORG", "LOC", "MISC"]
print("spaCy", spacy.__version__, {i: (n.meta["name"], n.meta["version"]) for i, n in NLP.items()})
print("Modelli caricati correttamente.")

---
## 1. POS e dipendenze

Un modello spaCy è una *pipeline*: tokenizza e poi aggiunge, token per token, la categoria grammaticale (`pos_`), il lemma (`lemma_`), la relazione sintattica con la sua testa (`dep_`) e, più avanti, le entità (`ents`). Guardiamo una frase breve per ciascuna lingua.

In [ ]:
# datos-es
FRASE_DEMO = {
    "spa": "Lucía Sevilla presentó ayer el plan en Bilbao.",
    "ita": "Lucia Salerno ha presentato ieri il piano a Bologna.",
}
FRASE_DOS_VERBOS = {
    "spa": "El alcalde inauguró la feria y los vecinos visitaron el museo.",
    "ita": "Il sindaco ha inaugurato la fiera e i cittadini hanno visitato il museo.",
}

In [ ]:
def tabla_tokens(doc):
    return pd.DataFrame([{"i": t.i, "token": t.text, "pos": t.pos_, "lemma": t.lemma_, "dep": t.dep_, "testa": t.head.text}
                         for t in doc])

docs_demo = {idioma: NLP[idioma](FRASE_DEMO[idioma]) for idioma in MODELOS}
for idioma, doc in docs_demo.items():
    print(idioma, "→", doc.text)
    display(tabla_tokens(doc))
print("nsubj =", spacy.explain("nsubj"), "| obj =", spacy.explain("obj"), "| ROOT =", spacy.explain("ROOT"))

# Controllo robusto: il verbo principale ha il lemma all'infinito
raiz = {i: [t for t in d if t.dep_ == "ROOT"][0] for i, d in docs_demo.items()}
assert raiz["spa"].lemma_ == "presentar" and raiz["ita"].lemma_ == "presentare"
print("Lemmi della radice:", {i: r.lemma_ for i, r in raiz.items()})

### Cosa osservi?

- La **radice** (`ROOT`) è il verbo. In italiano la forma composta `ha presentato` ha un ausiliare (`aux`) e la radice è il participio; in spagnolo il passato remoto `presentó` è una sola parola.
- I nomi propri (`Lucia Salerno`) formano un blocco: la seconda parola dipende dalla prima con una relazione `flat`.
- È la base di tutto ciò che segue: per «chi fa cosa» cercheremo il figlio `nsubj` e il figlio `obj` di ogni verbo.

### L'albero come struttura dati

Ogni token ha `head` (la sua testa) e `children` (i suoi dipendenti); `subtree` è il token con tutto ciò che dipende da lui. Stampiamo l'albero con rientro a partire dalla radice.

In [ ]:
def imprimir_arbol(token, nivel=0):
    print("    " * nivel + f"{token.text} [{token.dep_}]")
    for hijo in token.children:
        imprimir_arbol(hijo, nivel + 1)

for idioma, doc in docs_demo.items():
    print(idioma)
    imprimir_arbol(raiz[idioma])

# Proprietà dell'albero: ogni token è nel sottoalbero della radice della sua frase
for idioma, doc in docs_demo.items():
    assert len(list(raiz[idioma].subtree)) == len(doc)

---
## 2. Estrarre «chi fa cosa» dall'albero

L'idea: per ogni verbo, il suo soggetto è il figlio con `dep_` che inizia per `nsubj` e il suo oggetto è il figlio `obj`. Restituiamo il testo del **sottoalbero** di ciascuno, per conservare determinanti e nomi composti. Prima di fidarci del modello, proviamo la funzione con un albero **costruito a mano** (`Doc` con teste e dipendenze fissate): così l'`assert` verifica il nostro codice e non le previsioni del modello.

In [ ]:
def extraer_svo(doc):
    """Lista di (soggetto, lemma del verbo, oggetto) per ogni verbo che li abbia entrambi."""
    ternas = []
    for t in doc:
        if t.pos_ != "VERB":
            continue
        sujetos = [h for h in t.children if h.dep_.startswith("nsubj")]
        objetos = [h for h in t.children if h.dep_ == "obj"]
        if sujetos and objetos:
            sujeto = "".join(x.text_with_ws for x in sujetos[0].subtree).strip()
            objeto = "".join(x.text_with_ws for x in objetos[0].subtree).strip()
            ternas.append((sujeto, t.lemma_, objeto))
    return ternas

In [ ]:
# datos-es
from spacy.tokens import Doc
_v = spacy.blank("es").vocab
# Albero scritto a mano (teste assolute)
PALABRAS = ["La", "alcaldesa", "inauguró", "la", "feria", "y", "el", "concejal", "visitó", "el", "museo", "."]
doc_a_mano = Doc(_v, words=PALABRAS,
                 heads=[1, 2, 2, 4, 2, 8, 7, 8, 2, 10, 8, 2],
                 deps=["det", "nsubj", "ROOT", "det", "obj", "cc", "det", "nsubj", "conj", "det", "obj", "punct"],
                 pos=["DET", "NOUN", "VERB", "DET", "NOUN", "CCONJ", "DET", "NOUN", "VERB", "DET", "NOUN", "PUNCT"],
                 lemmas=["el", "alcaldesa", "inaugurar", "el", "feria", "y", "el", "concejal", "visitar", "el", "museo", "."])

In [ ]:
esperado = [("La alcaldesa", "inaugurar", "la feria"), ("el concejal", "visitar", "el museo")]     # dato-es
assert extraer_svo(doc_a_mano) == esperado, "extraer_svo non recupera le due terne dell'albero scritto a mano"
print("Albero a mano:", extraer_svo(doc_a_mano))

for idioma in MODELOS:
    doc = NLP[idioma](FRASE_DOS_VERBOS[idioma])
    print(idioma, FRASE_DOS_VERBOS[idioma])
    print("  terne:", extraer_svo(doc))
    print("  verbi:", [(t.text, t.dep_, t.head.text) for t in doc if t.pos_ == "VERB"])
    print("  soggetti:", [(t.text, t.dep_, t.head.text) for t in doc if t.dep_.startswith("nsubj")])

### Cosa osservi?

Con spaCy 3.8 e `es_core_news_sm` 3.8.0 si recuperano le due terne della frase con due verbi. Con `it_core_news_sm` 3.8.0 l'albero della stessa frase appende il secondo soggetto (`cittadini`) come coordinato dell'oggetto `fiera`, e il secondo verbo resta **senza soggetto**: la funzione restituisce una sola terna. L'estrattore non sbaglia: riceve un albero sbagliato. (Con un'altra versione del modello il risultato può cambiare.) Regola pratica: l'estrazione eredita gli errori dell'analisi sintattica, e un modello piccolo sbaglia soprattutto con coordinazioni e frasi lunghe.

---
## 3. NER e il corpus annotato

Per valutare un NER serve un **riferimento** (*gold*): frasi con le entità segnate da una persona. Abbiamo scritto 67 frasi di notizie sintetiche per lingua (persone inventate; luoghi e organizzazioni generici o reali, senza notizie false) e le abbiamo annotate **a mano** con sole quattro etichette (PER, ORG, LOC, MISC), le stesse che producono i modelli piccoli di spaCy. Ogni riga di `noticias_ner_es.csv` e `noticias_ner_it.csv` ha `id`, `texto` ed `entidades` (JSON con `[inizio, fine, etichetta]` in caratteri). I criteri completi sono in `datos/LEEME.md`.

In [ ]:
ARCHIVO_NER = {"spa": "noticias_ner_es.csv", "ita": "noticias_ner_it.csv"}      # dato-es

def cargar_corpus(idioma):
    """Legge il corpus NER annotato: lista di (id, testo, {(inizio, fine, etichetta)})."""
    df = pd.read_csv(cargar_datos(ARCHIVO_NER[idioma]))
    return [(r.id, r.texto, {(a, b, e) for a, b, e in json.loads(r.entidades)}) for r in df.itertuples()]

CORPUS = {idioma: cargar_corpus(idioma) for idioma in MODELOS}
for idioma, corpus in CORPUS.items():
    for _, texto, ents in corpus:
        assert all(0 <= a < b <= len(texto) for a, b, _ in ents), "scostamento fuori dal testo"
        assert all(e in ETIQUETAS for _, _, e in ents), "etichetta sconosciuta"
    assert len(corpus) >= 60
GOLD = {i: [g for _, _, g in c] for i, c in CORPUS.items()}
TEXTOS = {i: [t for _, t, _ in c] for i, c in CORPUS.items()}
IDS = {i: [x for x, _, _ in c] for i, c in CORPUS.items()}
conteo = pd.DataFrame({i: Counter(e for g in GOLD[i] for _, _, e in g) for i in CORPUS}).loc[ETIQUETAS]
conteo.loc["totale"] = conteo.sum()
print({i: len(c) for i, c in CORPUS.items()}, "frasi per lingua")
conteo

In [ ]:
for idioma in MODELOS:
    doc = NLP[idioma](FRASE_DEMO[idioma])
    print(idioma, [(e.text, e.label_, e.start_char, e.end_char) for e in doc.ents])
# Le entità di spaCy sono span di caratteri: entità = testo[inizio:fine]
doc = NLP["spa"](FRASE_DEMO["spa"])
assert all(doc.text[e.start_char:e.end_char] == e.text for e in doc.ents)

---
## 4. Valutare un NER: prima, cos'è un successo

Prima di calcolare qualcosa bisogna decidere **cosa conta come successo**. Un'entità prevista è una terna `(inizio, fine, etichetta)`. Tre definizioni ragionevoli:

| Modo | È un successo (TP) se… | Quando serve |
|---|---|---|
| `exacto` | lo span **e** l'etichetta coincidono | lo standard; il più severo |
| `tramo` | lo span coincide, l'etichetta si ignora | misurare solo la *localizzazione* |
| `parcial` | stessa etichetta e span **sovrapposti** | tollerare confini imprecisi |

Con qualunque di esse: **precisione** = TP/(TP+FP) (di ciò che ha detto il modello, quanto è corretto), **richiamo** (*recall*) = TP/(TP+FN) (di ciò che c'era, quanto ha trovato) e **F1** = media armonica. Un'entità prevista che non è un successo è un FP e un'entità del riferimento non trovata è un FN: **un errore di confine conta come FP e FN insieme** nel modo esatto. Useremo il modo `exacto` salvo indicazione contraria. Se un denominatore è 0 (un'etichetta senza previsioni, per esempio), per convenzione il valore è 0.

In [ ]:
def solapan(a, b):
    """Due span (inizio, fine, ...) si sovrappongono se condividono almeno un carattere."""
    return a[0] < b[1] and b[0] < a[1]

def coincide(a, b, modo):
    if modo == "exacto":                       # stesso span e stessa etichetta
        return a == b
    if modo == "tramo":                        # stesso span, etichetta ignorata
        return a[:2] == b[:2]
    return a[2] == b[2] and solapan(a, b)      # "parcial": stessa etichetta e span sovrapposti

def contar(gold, pred, modo="exacto"):
    """Restituisce {etichetta: [tp, fp, fn]}. gold e pred: una lista di insiemi {(inizio, fine, etichetta)} per frase."""
    cuenta = {e: [0, 0, 0] for e in ETIQUETAS}
    for g, p in zip(gold, pred):
        g, p = sorted(g), sorted(p)
        usados_g, usados_p = set(), set()
        for i, a in enumerate(g):
            for j, b in enumerate(p):
                if j not in usados_p and coincide(a, b, modo):
                    usados_g.add(i); usados_p.add(j); cuenta[a[2]][0] += 1
                    break
        for j, b in enumerate(p):
            if j not in usados_p:
                cuenta[b[2]][1] += 1
        for i, a in enumerate(g):
            if i not in usados_g:
                cuenta[a[2]][2] += 1
    return cuenta

def prf(tp, fp, fn):
    """Precisione, richiamo e F1; se il denominatore è 0, il valore è 0."""
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

def resumen(cuenta):
    tp, fp, fn = (sum(v[k] for v in cuenta.values()) for k in range(3))
    return prf(tp, fp, fn), (tp, fp, fn)

def tabla(cuenta):
    filas = []
    for e in ETIQUETAS:
        tp, fp, fn = cuenta[e]
        filas.append([e, tp, fp, fn, *prf(tp, fp, fn)])
    (p, r, f), (tp, fp, fn) = resumen(cuenta)
    filas.append(["TOTALE (micro)", tp, fp, fn, p, r, f])
    return pd.DataFrame(filas, columns=["etichetta", "tp", "fp", "fn", "precisione", "recall", "F1"]).round(3)

### Calcolare a mano un esempio piccolo

Quattro frasi; per ciascuna, il riferimento e la previsione di un modello **immaginario** (con `{testo|ETICHETTA}` segniamo le entità). Conta prima di eseguire: nel modo esatto ci sono 4 successi (`Anna Rossi`, `Bologna`, `Ginevra`, `Roma`), 3 FP (`ONU` con etichetta sbagliata, `Croce` con confine sbagliato, `Ieri`) e 3 FN (`ONU` come ORG, `Croce Rossa`, `italiani`). Quindi P = 4/7, R = 4/7 e F1 = 4/7 ≈ 0,571.

In [ ]:
# datos-es
# Esempio minimo da calcolare a mano: ogni frase con il riferimento (gold) e la previsione di un modello immaginario
MANO = {
    "spa": [("Ana Ruiz vive en Bilbao.", "{Ana Ruiz|PER} vive en {Bilbao|LOC}.", "{Ana Ruiz|PER} vive en {Bilbao|LOC}."),
            ("La ONU se reunió en Ginebra con la Cruz Roja.", "La {ONU|ORG} se reunió en {Ginebra|LOC} con la {Cruz Roja|ORG}.", "La {ONU|LOC} se reunió en {Ginebra|LOC} con la {Cruz|ORG} Roja."),
            ("Los italianos llegaron ayer.", "Los {italianos|MISC} llegaron ayer.", "Los italianos llegaron ayer."),
            ("Ayer llovió en Roma.", "Ayer llovió en {Roma|LOC}.", "{Ayer|PER} llovió en {Roma|LOC}.")],
    "ita": [("Anna Rossi vive a Bologna.", "{Anna Rossi|PER} vive a {Bologna|LOC}.", "{Anna Rossi|PER} vive a {Bologna|LOC}."),
            ("L'ONU si è riunita a Ginevra con la Croce Rossa.", "L'{ONU|ORG} si è riunita a {Ginevra|LOC} con la {Croce Rossa|ORG}.", "L'{ONU|LOC} si è riunita a {Ginevra|LOC} con la {Croce|ORG} Rossa."),
            ("Gli italiani sono arrivati ieri.", "Gli {italiani|MISC} sono arrivati ieri.", "Gli italiani sono arrivati ieri."),
            ("Ieri è piovuto a Roma.", "Ieri è piovuto a {Roma|LOC}.", "{Ieri|PER} è piovuto a {Roma|LOC}.")],
}

In [ ]:
for idioma in MODELOS:
    gold_m, pred_m = [], []
    for _, ref, pred in MANO[idioma]:
        t1, g = parsear(ref)
        t2, p = parsear(pred)
        assert t1 == t2, "il riferimento e la previsione devono avere lo stesso testo"
        gold_m.append(g); pred_m.append(p)
    resultados = {modo: resumen(contar(gold_m, pred_m, modo)) for modo in ("exacto", "tramo", "parcial")}
    # Cifre calcolate a mano nel testo
    assert resultados["exacto"][1] == (4, 3, 3)
    assert np.isclose(resultados["exacto"][0][2], 4 / 7)
    assert resultados["parcial"][1] == (5, 2, 2) and resultados["tramo"][1] == (5, 2, 2)
    cuenta = contar(gold_m, pred_m)
    assert cuenta["PER"] == [1, 1, 0] and cuenta["LOC"] == [3, 1, 0] and cuenta["ORG"] == [0, 1, 2] and cuenta["MISC"] == [0, 0, 1]
    print(idioma, {m: (r[1], tuple(round(x, 3) for x in r[0])) for m, r in resultados.items()})
tabla(contar(gold_m, pred_m))

### Cosa osservi?

- La stessa previsione dà **F1 = 0,571 in modo esatto** e valori più alti in `tramo` e `parcial`: scegliere il modo è una decisione di valutazione, e va dichiarata insieme alla cifra.
- L'etichetta `ORG` ha precisione e richiamo 0 (nessun successo), e `MISC` ha precisione 0 per convenzione (0/0) e richiamo 0: per questo conviene guardare **per etichetta** e non solo il totale.
- Il totale «micro» somma TP, FP e FN di tutte le etichette prima di dividere; la media «macro» farebbe la media degli F1 per etichetta e dà lo stesso peso alle etichette rare.

---
## 5. Esperimento guidato: come se la cava il modello piccolo?

Facciamo passare le 67 frasi di ciascuna lingua dal loro modello e confrontiamo con il riferimento, nei tre modi e per etichetta. **Prevedi prima di eseguire:** quale etichetta credi sarà la più difficile? In quale modo sarà più alta la F1?

In [ ]:
PRED = {i: [{(e.start_char, e.end_char, e.label_) for e in doc.ents} for doc in NLP[i].pipe(TEXTOS[i])] for i in MODELOS}
resultados = {}
for idioma in MODELOS:
    for modo in ("exacto", "tramo", "parcial"):
        resultados[idioma, modo] = contar(GOLD[idioma], PRED[idioma], modo)

n_gold = {i: sum(len(g) for g in GOLD[i]) for i in MODELOS}
for idioma in MODELOS:
    tp, fp, fn = resumen(resultados[idioma, "exacto"])[1]
    assert tp + fn == n_gold[idioma], "TP + FN deve essere il numero di entità del riferimento"
    assert tp + fp == sum(len(p) for p in PRED[idioma]), "TP + FP deve essere il numero di entità previste"
    assert resumen(resultados[idioma, "exacto"])[1][0] <= resumen(resultados[idioma, "parcial"])[1][0] <= n_gold[idioma]

comparacion = pd.DataFrame({(idioma, modo): dict(zip(["precisione", "recall", "F1"], resumen(resultados[idioma, modo])[0]))
                            for idioma in MODELOS for modo in ("exacto", "tramo", "parcial")}).T.round(3)
display(comparacion)
for idioma in MODELOS:
    print(idioma, "modo esatto, per etichetta")
    display(tabla(resultados[idioma, "exacto"]))

In [ ]:
f1_et = pd.DataFrame({idioma: [prf(*resultados[idioma, "exacto"][e])[2] for e in ETIQUETAS] for idioma in MODELOS}, index=ETIQUETAS)
ax = f1_et.plot.bar(rot=0, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("F1 (modo esatto)")
ax.set_ylim(0, 1)
ax.set_title("F1 per etichetta con spaCy 3.8 (esempio illustrativo)")
ax.legend(["es_core_news_sm", "it_core_news_sm"])
plt.show()
assert f1_et.shape == (4, 2) and ((f1_et >= 0) & (f1_et <= 1)).all().all()

### Cosa osservi?

Con spaCy 3.8 e i modelli 3.8.0, su questo corpus (*esempio illustrativo*, F1 micro in modo esatto: circa 0,70 in spagnolo e 0,76 in italiano):

- La F1 sale parecchio passando da `exacto` a `tramo` (≈ 0,90 e ≈ 0,88): **molte entità sono localizzate bene ma con un'altra etichetta**.
- `PER` e `LOC` vanno meglio di `ORG` e `MISC`; `ORG` ha precisione alta e richiamo basso: il modello non la prevede quasi mai dove dovrebbe.
- La differenza tra lingue viene da due modelli diversi e da due corpus che sono adattamenti, non traduzioni esatte: **non si può dire che l'italiano sia «più facile»**.
- Con 144 entità per lingua, una sola entità sposta la F1 di circa 0,7 punti percentuali: le differenze piccole non sono significative.

---
## 6. Errori tipici di un modello piccolo

Classifichiamo ogni errore del modo esatto in quattro tipi:

| Tipo | Significato |
|---|---|
| `etiqueta` | stesso span, etichetta diversa (ad esempio ORG → LOC) |
| `frontera` | lo span si sovrappone a un altro ma non coincide |
| `omitida` | entità del riferimento senza alcuna previsione sovrapposta |
| `espuria` | previsione che non si sovrappone a nessuna entità del riferimento |

In [ ]:
def tipos_error(gold, pred):
    tipos = Counter()
    for g, p in zip(gold, pred):
        for a in g - p:                                    # entità del riferimento non riuscite
            if any(a[:2] == b[:2] for b in p):
                tipos["etiqueta"] += 1
            elif any(solapan(a, b) for b in p):
                tipos["frontera"] += 1
            else:
                tipos["omitida"] += 1
        for b in p - g:                                    # previsioni che non sono nel riferimento
            if not any(solapan(a, b) for a in g):
                tipos["espuria"] += 1
    return tipos

errores = {i: tipos_error(GOLD[i], PRED[i]) for i in MODELOS}
for idioma in MODELOS:
    fn = resumen(resultados[idioma, "exacto"])[1][2]
    assert errores[idioma]["etiqueta"] + errores[idioma]["frontera"] + errores[idioma]["omitida"] == fn, "i tre tipi devono sommare i FN"
display(pd.DataFrame(errores).fillna(0).astype(int).reindex(["etiqueta", "frontera", "omitida", "espuria"]))

confusion = {i: Counter((a[2], b[2]) for g, p in zip(GOLD[i], PRED[i]) for a in g for b in p if a[:2] == b[:2] and a[2] != b[2]) for i in MODELOS}
for idioma in MODELOS:
    print(idioma, "(etichetta corretta → prevista):", confusion[idioma].most_common(4))

In [ ]:
# Casi difficili del corpus: nome di persona che coincide con un luogo, sigle e gentili in minuscolo
def ver(idioma, prefijo_id):
    i = next(k for k, x in enumerate(IDS[idioma]) if x.endswith(prefijo_id))
    print(TEXTOS[idioma][i])
    print("   gold:", [(TEXTOS[idioma][i][a:b], e) for a, b, e in sorted(GOLD[idioma][i])])
    print("   spaCy:", [(TEXTOS[idioma][i][a:b], e) for a, b, e in sorted(PRED[idioma][i])])

for idioma in MODELOS:
    for pid in ("006", "010", "016"):
        ver(idioma, pid)
    print()

### Cosa osservi?

- L'errore più frequente in entrambe le lingue è di **etichetta**, soprattutto ORG → LOC: i modelli piccoli trattano come luogo un'organizzazione con nome dall'aspetto geografico o sconosciuto (`Cooperativa Mare Nord`, `Ospedale Maggiore`).
- I **gentili in minuscolo** (`italiani`, `portoghesi`) restano spesso senza etichetta: la maiuscola iniziale è un indizio forte per questi modelli.
- Le parole con la maiuscola per l'inizio di frase (`Según`, `Ayer`, `Ieri`) generano qualche entità spuria.
- Altri errori di confine: il modello include una parola generica (`fiume Po`, `porto di Genova`) o taglia un nome lungo.

- I casi ambigui persona/luogo (`Carlo Siena è andato a Siena`) sono risolti bene da questo modello: la difficoltà è altrove.

Non sono «guasti rari»: sono il segnale di quali regole scrivere adesso.

---
## 7. Regole che correggono il modello: `Matcher` ed `EntityRuler`

- Il **`Matcher`** cerca **pattern di token** (`LOWER`, `IS_TITLE`, `POS`, `OP`…) e restituisce span; non cambia le entità da solo.
- L'**`EntityRuler`** è un componente che **assegna entità** con pattern; posto dopo `ner` e con `overwrite_ents=True`, le sue regole prevalgono sul modello.

Prima il `Matcher` con un pattern «carica + nome proprio» che usa il POS del modello.

In [ ]:
# datos-es
CARGOS = {"spa": ["alcalde", "alcaldesa", "doctora", "doctor", "presidente", "periodista", "actriz", "chef", "entrenador"],
          "ita": ["sindaco", "dottoressa", "dottore", "presidente", "giornalista", "attrice", "chef", "direttore"]}

In [ ]:
from spacy.matcher import Matcher
from spacy.util import filter_spans

def buscar_cargos(nlp, texto, cargos):
    matcher = Matcher(nlp.vocab)
    matcher.add("CARGO_NOMBRE", [[{"LOWER": {"IN": cargos}}, {"POS": "PROPN", "OP": "+"}]])
    doc = nlp(texto)
    return [s.text for s in filter_spans([doc[a:b] for _, a, b in matcher(doc)])]        # filter_spans tiene lo span più lungo

for idioma in MODELOS:
    hallazgos = [(x, buscar_cargos(NLP[idioma], t, CARGOS[idioma])) for x, t in zip(IDS[idioma], TEXTOS[idioma])]
    hallazgos = [h for h in hallazgos if h[1]]
    print(idioma, len(hallazgos), "frasi con «carica + nome»; esempi:", hallazgos[:3])

# Verifica con una frase scritta per il test: la regola usa il POS, quindi verifichiamo solo che il Matcher restituisca span di testo
_t = buscar_cargos(NLP["spa"], "El alcalde Pedro Soler habló ayer.", CARGOS["spa"])       # dato-es
assert all(isinstance(x, str) for x in _t)

### Un `EntityRuler` con regole di sviluppo

Per non ingannarci, dividiamo ogni corpus: le **prime 40 frasi** sono di *sviluppo* (guardiamo i loro errori e scriviamo le regole) e le **27 restanti** di *test* (non le tocchiamo mentre scriviamo le regole). Le regole sono tre: sigle di organizzazioni, gentili e un pattern `ORG` = nome di istituzione con maiuscola + nome proprio (con eventuali preposizioni). Le liste vengono dagli errori di sviluppo.

In [ ]:
# datos-es
SIGLAS = {"spa": ["ONU", "FMI", "OMS", "UGT"], "ita": ["ONU", "FMI", "OMS", "CGIL"]}
GENTILICIOS = {"spa": ["italianos", "portugueses", "gallega"], "ita": ["italiani", "portoghesi", "ligure"]}
NOMBRES_ORG = {"spa": ["cooperativa", "banco", "universidad", "hospital", "comisión", "gobierno", "federación", "fundación", "ayuntamiento", "óptica"],
               "ita": ["cooperativa", "banca", "università", "ospedale", "commissione", "regione", "federazione", "fondazione", "comune", "politecnico", "ottica"]}
PREPOSICIONES = ["de", "del", "la", "di", "della", "dei", "dell'"]

In [ ]:
def patrones(idioma):
    nombre = {"LOWER": {"IN": NOMBRES_ORG[idioma]}, "IS_TITLE": True}
    titulo = {"IS_TITLE": True, "OP": "+"}
    prep = {"LOWER": {"IN": PREPOSICIONES}}
    pats = [{"label": "ORG", "pattern": [{"ORTH": s}]} for s in SIGLAS[idioma]]
    pats.append({"label": "MISC", "pattern": [{"LOWER": {"IN": GENTILICIOS[idioma]}}]})
    for esquema in ([nombre, titulo], [nombre, prep, titulo], [nombre, titulo, prep, titulo], [nombre, prep, titulo, prep, titulo]):
        pats.append({"label": "ORG", "pattern": esquema})
    return pats

def con_reglas(idioma):
    nlp = spacy.load(MODELOS[idioma])
    ruler = nlp.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
    ruler.add_patterns(patrones(idioma))
    return nlp

NLP_R = {i: con_reglas(i) for i in MODELOS}
print({i: n.pipe_names for i, n in NLP_R.items()})

# Verifica robusta: le regole prevalgono sul modello
e_es = {e.text: e.label_ for e in NLP_R["spa"]("La ONU y los italianos llegaron a la reunión.").ents}       # dato-es
e_it = {e.text: e.label_ for e in NLP_R["ita"]("L'ONU e gli italiani sono arrivati alla riunione.").ents}      # dato-es
assert e_es["ONU"] == "ORG" and e_es["italianos"] == "MISC"
assert e_it["ONU"] == "ORG" and e_it["italiani"] == "MISC"

In [ ]:
DESARROLLO = slice(0, 40)
PRUEBA = slice(40, None)
PRED_R = {i: [{(e.start_char, e.end_char, e.label_) for e in doc.ents} for doc in NLP_R[i].pipe(TEXTOS[i])] for i in MODELOS}

filas = []
for idioma in MODELOS:
    for nombre, parte in (("sviluppo", DESARROLLO), ("test", PRUEBA)):
        for sistema, pred in (("solo modello", PRED), ("modello + regole", PRED_R)):
            (p, r, f), (tp, fp, fn) = resumen(contar(GOLD[idioma][parte], pred[idioma][parte]))
            assert tp + fn == sum(len(g) for g in GOLD[idioma][parte])
            filas.append([idioma, nombre, sistema, tp, fp, fn, round(p, 3), round(r, 3), round(f, 3)])
pd.DataFrame(filas, columns=["lingua", "partizione", "sistema", "tp", "fp", "fn", "precisione", "recall", "F1"])

### Cosa osservi?

Con spaCy 3.8 (*esempio illustrativo*): le regole fanno salire la F1 in **sviluppo** (da ≈ 0,70 a ≈ 0,80 in spagnolo e da ≈ 0,70 a ≈ 0,82 in italiano), ma nel **test** non si nota alcun miglioramento (le cifre prima e dopo non cambiano): le liste vengono da ciò che abbiamo visto in sviluppo (`ONU`, `italiani`…) e nel test compaiono altre sigle e gentili (`UE`, `sardi`…). È il rischio di scrivere regole guardando gli errori: **un miglioramento sulle frasi usate per scriverle non dice nulla sulle altre**. Bisogna misurare sempre su frasi tenute da parte, ampliare le regole con criterio (o con più dati) e verificare che non rompano successi precedenti.

---
## Riepilogo

| Concetto | Idea chiave |
|---|---|
| POS e dipendenze | l'albero collega ogni parola alla sua testa; `nsubj` e `obj` danno «chi fa cosa» |
| Estrazione SVO | eredita gli errori dell'analisi; verificare il codice con un albero scritto a mano |
| NER | entità come span `(inizio, fine, etichetta)` di caratteri |
| Valutazione | definire prima il successo (esatto, span, parziale); P, R e F1 **per etichetta** e in totale |
| Errori tipici | etichetta (ORG → LOC), confine, omissione ed entità spurie |
| Regole | `Matcher` cerca; `EntityRuler` assegna; misurare su frasi non usate per scriverle |

**Prossima:** nella lezione 8 usciamo dalla struttura e studiamo i **modelli di linguaggio n-gramma**: come assegnare probabilità a una sequenza di parole. Poi la pratica di questa lezione.